In [ ]:
import numpy as np
import pandas as pd
import glob
from warnings import filterwarnings
filterwarnings('ignore')

# ===============================================
# A) DATA LOADING (NEW: Feature-Engineered Dataset)
# ===============================================

print("--- Loading NEW Feature-Engineered Dataset... ---")

DATA_PATH = 'FeatureEngineering/atp_featured_dataset.npz'

try:
    # 1. Load saved NumPy arrays
    data = np.load(DATA_PATH, allow_pickle=True)
    
    # 2. Extract Variables
    X_train = data['X_train']
    Y_train = data['Y_train']
    X_val = data['X_val']
    Y_val = data['Y_val']
    X_test = data['X_test']
    Y_test = data['Y_test']
    
    # Feature names
    FINAL_FEATURES = data['feature_names'].tolist()
    
    # Scaler information (optional - for information purposes)
    scaler_mean = data['scaler_mean']
    scaler_scale = data['scaler_scale']

    # 3. Set Main Training Variables
    X = X_train
    Y = Y_train
    
    # Combine train + val for final training
    X_final_train = np.concatenate([X_train, X_val], axis=1)
    Y_final_train = np.concatenate([Y_train, Y_val], axis=1)
    
    # Define N_X here so CONFIGURATION block can use it
    N_X_loaded = X_train.shape[0] 
    
    print("--- Data loaded successfully. ---")
    
    # 4. CHECK (SUMMARY)
    print(f"\n--- Dataset Summary ---")
    print(f"Input Feature Count (N_X): {N_X_loaded}")
    print(f"Feature Names (first 10): {FINAL_FEATURES[:37]}")
    print("-" * 30)
    print(f"Phase 1 (Train): {X_train.shape[1]} samples (Variables: X, Y)")
    print(f"Phase 1 (Val):   {X_val.shape[1]} samples (Variables: X_val, Y_val)")
    print(f"Phase 2 (Final):  {X_final_train.shape[1]} samples (Variables: X_final_train, Y_final_train)")
    print("-" * 30)
    print(f"Test Set:        {X_test.shape[1]} samples (Variables: X_test, Y_test)")
    print("\n⚠️  IMPORTANT: Data is normalized with StandardScaler (mean=0, std=1)")
    print("   This is OPTIMAL for L2 regularization and He initialization!")

except FileNotFoundError:
    print("\n--- ERROR: 'atp_featured_dataset.npz' file not found! ---")
    print("Please make sure you have run the '04_create_npz_dataset_FIXED.py' script first.")
    X, Y, X_val, Y_val, X_test, Y_test, X_final_train, Y_final_train, FINAL_FEATURES, N_X_loaded = (None,)*10
except Exception as e:
    print(f"\n--- An unexpected error occurred while loading data: {e} ---")
    X, Y, X_val, Y_val, X_test, Y_test, X_final_train, Y_final_train, FINAL_FEATURES, N_X_loaded = (None,)*10


In [ ]:
# ===============================================
# CONFIGURATION: HYPER-PARAMETERS (FOR NEW FEATURE-ENGINEERED DATASET)
# ===============================================
N_X = N_X_loaded        # Input Size (Now 37 features!)

# Model architecture - optimized for 37 features
N_H1 = 128              # First hidden layer (larger for more features)
N_H2 = 64               # Second hidden layer
N_Y = Y.shape[0]        # Output Size (1)

LEARNING_RATE = 0.003   # Slightly lower (more features)
EPOCHS = 5000          
PATIENCE = 10           # Slightly more patient (more complex model)
FINAL_EPOCHS = None

# --- ADAM OPTIMIZER PARAMETERS ---
BETA1 = 0.9     
BETA2 = 0.999   
EPSILON = 1e-8

# L2 Regularization
LAMBDA = 0.5            # Lower lambda is sufficient since we use StandardScaler

# Model Choice
MODEL_CHOICE = None

# UPDATED CONTROL OUTPUT
print(f"--- Configuring Architecture (Feature-Engineered Dataset) ---")
print(f"Input Size (N_X): {N_X} (51 engineered features!)")
print(f"Hidden Layer 1 (N_H1): {N_H1}")
print(f"Hidden Layer 2 (N_H2): {N_H2}")
print(f"Learning Rate: {LEARNING_RATE} | L2 Penalty (Lambda): {LAMBDA}")
print(f"\n✅ StandardScaler normalization active (mean=0, std=1)")
print(f"✅ OPTIMAL settings for He initialization + Leaky ReLU")
print(f"✅ Expected accuracy improvement: +10-18% (compared to basic features)")


In [ ]:
# ===============================================
# A) (HELPER FUNCTIONS)
# ===============================================

def sigmoid(Z):
    """Sigmoid activation function."""
    A = 1 / (1 + np.exp(-Z))
    return A


# ===============================================
# B) (MODEL FUNCTIONS)
# ===============================================

# --- Model 1: Tanh / Xavier (2 Layer) ---

def initialize_parameters_Xavier(n_x, n_h1, n_h2, n_y):
    """ Initialize parameters for a 2-layer network with Xavier initialization. """
    W1 = np.random.randn(n_h1, n_x) * np.sqrt(1 / n_x)
    b1 = np.zeros((n_h1, 1))
    W2 = np.random.randn(n_h2, n_h1) * np.sqrt(1 / n_h1) # H1 -> H2
    b2 = np.zeros((n_h2, 1))
    W3 = np.random.randn(n_y, n_h2) * np.sqrt(1 / n_h2) # H2 -> Output
    b3 = np.zeros((n_y, 1))
    
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}
    return parameters

def forward_propagation_tanh(X, parameters):
    """ Forward propagation for a 2-layer network with tanh activation. """
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]
    
    # Layer 1 (Input -> H1)
    Z1 = np.dot(W1, X) + b1
    A1 = np.tanh(Z1) 
    # Layer 2 (H1 -> H2)
    Z2 = np.dot(W2, A1) + b2
    A2 = np.tanh(Z2) 
    # Layer 3 (H2 -> Output)
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3) # Final prediction
    
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3}
    return A3, cache # Final prediction A3

# --- Model 2: Leaky ReLU / He (2 Layer) ---

def initialize_parameters_He(n_x, n_h1, n_h2, n_y):
    """ Initialize parameters for a 2-layer network with He initialization. """
    W1 = np.random.randn(n_h1, n_x) * np.sqrt(2 / n_x)
    b1 = np.zeros((n_h1, 1))
    W2 = np.random.randn(n_h2, n_h1) * np.sqrt(2 / n_h1) # H1 -> H2
    b2 = np.zeros((n_h2, 1))
    W3 = np.random.randn(n_y, n_h2) * np.sqrt(2 / n_h2) # H2 -> Output
    b3 = np.zeros((n_y, 1))
    
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}
    return parameters

def forward_propagation_LeakyReLU(X, parameters):
    """ Forward propagation for a 2-layer network with Leaky ReLU activation. """
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]
    
    # Layer 1 (Input -> H1)
    Z1 = np.dot(W1, X) + b1
    A1 = np.maximum(0.01 * Z1, Z1) # LReLU
    # Layer 2 (H1 -> H2)
    Z2 = np.dot(W2, A1) + b2
    A2 = np.maximum(0.01 * Z2, Z2) # LReLU
    # Layer 3 (H2 -> Output)
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3) # Final prediction
    
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3}
    return A3, cache # Final prediction is A3

# ===============================================
# D) COST FUNCTION (GENERIC L2 REGULARIZATION)
# ===============================================

def compute_cost(A_out, Y, parameters, lambd=0): 
    """
    Computes Binary Cross-Entropy and generic L2 regularization penalty.
    A_out: Final activation of the network (A2, A3, etc.)
    """
    m = Y.shape[1] 
    
    # 1. Cross-Entropy Cost
    epsilon = 1e-8 
    A_clipped = np.clip(A_out, epsilon, 1 - epsilon)
    log_probs = Y * np.log(A_clipped)
    log_probs_complement = (1 - Y) * np.log(1 - A_clipped)
    cross_entropy_cost = - (1 / m) * (np.sum(log_probs + log_probs_complement))
    
    # 2. L2 Regularization Penalty (Generic)
    # Applies to all weights (W1, W2, W3...) in the parameters dictionary
    L2_cost = 0
    L = len(parameters) // 2  # Number of layers (W,b pairs)
    for l in range(1, L + 1):
        L2_cost += np.sum(np.square(parameters["W" + str(l)]))
        
    L2_regularization_cost = (lambd / (2 * m)) * L2_cost
    
    cost = cross_entropy_cost + L2_regularization_cost
    cost = np.squeeze(cost) 
    
    return cost


# ===============================================
# E) BACKWARD PROPAGATION (2 LAYER)
# ===============================================

def backward_propagation_tanh(parameters, cache, X, Y, lambd=0):
    """ Backward propagation for a 2-layer network with tanh activation. """
    m = X.shape[1]
    # Get activations from cache
    A1, A2, A3 = cache["A1"], cache["A2"], cache["A3"]
    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    
    # Layer 3 (Output)
    dZ3 = A3 - Y
    dW3 = (1 / m) * np.dot(dZ3, A2.T) + (lambd / m) * W3
    db3 = np.mean(dZ3, axis=1, keepdims=True)
    
    # Layer 2 (Hidden)
    dZ2_weighted = np.dot(W3.T, dZ3)
    dZ2 = dZ2_weighted * (1 - np.power(A2, 2))
    dW2 = (1 / m) * np.dot(dZ2, A1.T) + (lambd / m) * W2
    db2 = np.mean(dZ2, axis=1, keepdims=True)
    
    # Layer 1 (Hidden)
    dZ1_weighted = np.dot(W2.T, dZ2)
    dZ1 = dZ1_weighted * (1 - np.power(A1, 2))
    dW1 = (1 / m) * np.dot(dZ1, X.T) + (lambd / m) * W1
    db1 = np.mean(dZ1, axis=1, keepdims=True)
    
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2, "dW3": dW3, "db3": db3}
    return grads

def backward_propagation_LeakyReLU(parameters, cache, X, Y, lambd=0):
    """ Backward propagation for a 2-layer network with Leaky ReLU activation. """
    m = X.shape[1]
    A1, A2, A3 = cache["A1"], cache["A2"], cache["A3"]
    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    
    # Layer 3 (Output)
    dZ3 = A3 - Y
    dW3 = (1 / m) * np.dot(dZ3, A2.T) + (lambd / m) * W3
    db3 = np.mean(dZ3, axis=1, keepdims=True)
    
    # Layer 2 (Hidden)
    dZ2_weighted = np.dot(W3.T, dZ3)
    dZ2_act = np.where(A2 > 0, 1.0, 0.01)
    dZ2 = dZ2_weighted * dZ2_act
    dW2 = (1 / m) * np.dot(dZ2, A1.T) + (lambd / m) * W2
    db2 = np.mean(dZ2, axis=1, keepdims=True)
    
    # Layer 1 (Hidden)
    dZ1_weighted = np.dot(W2.T, dZ2)
    dZ1_act = np.where(A1 > 0, 1.0, 0.01)
    dZ1 = dZ1_weighted * dZ1_act
    dW1 = (1 / m) * np.dot(dZ1, X.T) + (lambd / m) * W1
    db1 = np.mean(dZ1, axis=1, keepdims=True)
    
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2, "dW3": dW3, "db3": db3}
    return grads

# ========================
# F) UPDATING PARAMETERS
# ========================

def update_parameters_gradient_descent(parameters, grads, learning_rate):
    """ Generic (L-layer) network gradient descent update. """
    L = len(parameters) // 2 # Number of layers (2 or 3)
    
    for l in range(1, L + 1):
        parameters["W" + str(l)] = parameters["W" + str(l)] - learning_rate * grads["dW" + str(l)]
        parameters["b" + str(l)] = parameters["b" + str(l)] - learning_rate * grads["db" + str(l)]
        
    return parameters

# --- ADAM OPTIMIZATION FUNCTIONS (NO CHANGES NEEDED) ---
# These functions are already generic (L-layer) written.
# They will work correctly for L=3 (W1,W2,W3).

def initialize_adam(parameters):
    v = {}
    s = {}
    L = len(parameters) // 2 
    for l in range(1, L + 1):
        v["dW" + str(l)] = np.zeros(parameters["W" + str(l)].shape)
        v["db" + str(l)] = np.zeros(parameters["b" + str(l)].shape)
        s["dW" + str(l)] = np.zeros(parameters["W" + str(l)].shape)
        s["db" + str(l)] = np.zeros(parameters["b" + str(l)].shape)
    return v, s

def update_parameters_adam(parameters, grads, v, s, t, learning_rate, beta1, beta2, epsilon):
    L = len(parameters) // 2 
    v_corrected = {} 
    s_corrected = {} 
    
    for l in range(1, L + 1):
        dW = grads["dW" + str(l)]
        db = grads["db" + str(l)]
        
        # Update v and s
        v["dW" + str(l)] = beta1 * v["dW" + str(l)] + (1 - beta1) * dW
        v["db" + str(l)] = beta1 * v["db" + str(l)] + (1 - beta1) * db
        s["dW" + str(l)] = beta2 * s["dW" + str(l)] + (1 - beta2) * np.power(dW, 2)
        s["db" + str(l)] = beta2 * s["db" + str(l)] + (1 - beta2) * np.power(db, 2)
        
        # Bias Correction
        v_corrected["dW" + str(l)] = v["dW" + str(l)] / (1 - np.power(beta1, t))
        v_corrected["db" + str(l)] = v["db" + str(l)] / (1 - np.power(beta1, t))
        s_corrected["dW" + str(l)] = s["dW" + str(l)] / (1 - np.power(beta2, t))
        s_corrected["db" + str(l)] = s["db" + str(l)] / (1 - np.power(beta2, t))
        
        # Update parameters
        W_update = (learning_rate * v_corrected["dW" + str(l)]) / (np.sqrt(s_corrected["dW" + str(l)]) + epsilon)
        b_update = (learning_rate * v_corrected["db" + str(l)]) / (np.sqrt(s_corrected["db" + str(l)]) + epsilon)
        
        parameters["W" + str(l)] = parameters["W" + str(l)] - W_update
        parameters["b" + str(l)] = parameters["b" + str(l)] - b_update

    return parameters, v, s

# ========================
# G) TRAINING FUNCTIONS
# ========================

FINAL_EPOCHS = 0

# --- Model 1: Tanh / Xavier (2 Layer) ---
def train_ann_tanh_xavier(X, Y, n_h1, n_h2, learning_rate, num_epochs, # n_h -> n_h1, n_h2
              X_val=None, Y_val=None, 
              print_cost=False, patience=3,
              lambd=0): # L2 added
    
    n_x, n_y = X.shape[0], Y.shape[0] 
    
    global FINAL_EPOCHS
    
    # 2 Layer initialization
    parameters = initialize_parameters_Xavier(n_x, n_h1, n_h2, n_y)
    
    costs, validation_costs = [], []
    best_val_cost, patience_counter, best_epoch = np.inf, 0, 0
    best_parameters = {}
    
    do_validation = X_val is not None and Y_val is not None
    print_interval = num_epochs / 100 if (num_epochs / 100) >= 1 else 1
    
    if not do_validation:
        patience = np.inf 
        if print_cost:
            print("--- WARNING: Validation set is missing. Early Stopping disabled. ---")

    for i in range(num_epochs):
        
        # Call 2 Layer functions
        A_out, cache = forward_propagation_tanh(X, parameters)
        cost = compute_cost(A_out, Y, parameters, lambd) # A_out = A3
        grads = backward_propagation_tanh(parameters, cache, X, Y, lambd)
        parameters = update_parameters_gradient_descent(parameters, grads, learning_rate)
        
        if i % print_interval == 0:
            costs.append(cost) 
            
            if do_validation:
                A_val, _ = forward_propagation_tanh(X_val, parameters) 
                cost_val = compute_cost(A_val, Y_val, parameters, lambd) 
                validation_costs.append(cost_val) 
                
                if print_cost:
                    print(f"Epoch {i} | Training Cost: {cost:.4f} | Validation Cost: {cost_val:.4f}")
                
                if cost_val < best_val_cost:
                    best_val_cost, best_parameters, best_epoch, patience_counter = cost_val, parameters.copy(), i, 0
                    if print_cost: print(f"          -> (The best model has been saved.)")
                    FINAL_EPOCHS = best_epoch
                else:
                    patience_counter += 1
                    if print_cost: print(f"          -> (Patience: {patience_counter}/{patience})")
                
                if patience_counter >= patience:
                    print(f"\n--- Early Stopping: {best_epoch} (Validation Cost: {best_val_cost:.4f})")
                    break 
            else:
                validation_costs.append(np.nan) 
                if print_cost: print(f"Epoch {i} | Training Cost: {cost:.4f}")
            
    if do_validation and best_parameters:
        return best_parameters, costs, validation_costs
    
    return parameters, costs, validation_costs

# --- Model 2: Leaky ReLU / He / Adam (2 Layer) ---
def train_ann_LeakyReLU_He(X, Y, n_h1, n_h2, learning_rate, num_epochs, # n_h -> n_h1, n_h2
              X_val=None, Y_val=None, 
              print_cost=False, patience=3,
              beta1=BETA1, beta2=BETA2, epsilon=EPSILON,
              lambd=0): 
    
    global FINAL_EPOCHS
    
    n_x, n_y = X.shape[0], Y.shape[0] 
    
    # 2 Layer initialization
    parameters = initialize_parameters_He(n_x, n_h1, n_h2, n_y)
    v, s = initialize_adam(parameters) # Adam (L=3)
    t = 0 
    
    costs, validation_costs = [], []
    best_val_cost, patience_counter, best_epoch = np.inf, 0, 0
    best_parameters = {}
    
    do_validation = X_val is not None and Y_val is not None
    print_interval = num_epochs / 100 if (num_epochs / 100) >= 1 else 1
    
    if not do_validation:
        patience = np.inf 
        if print_cost:
            print("--- WARNING: Validation set is missing. Early Stopping disabled. ---")

    for i in range(num_epochs):
        
        # 2 Layer functions
        A_out, cache = forward_propagation_LeakyReLU(X, parameters)
        cost = compute_cost(A_out, Y, parameters, lambd) # A_out = A3
        grads = backward_propagation_LeakyReLU(parameters, cache, X, Y, lambd)
        t += 1
        parameters, v, s = update_parameters_adam(parameters, grads, v, s, t, 
                                             learning_rate, beta1, beta2, epsilon)
        
        if i % 10 == 0:
            costs.append(cost) 
            
            if do_validation:
                A_val, _ = forward_propagation_LeakyReLU(X_val, parameters) 
                cost_val = compute_cost(A_val, Y_val, parameters, lambd) 
                validation_costs.append(cost_val) 
                
                if print_cost:
                    print(f"Epoch {i} | Training Cost: {cost:.4f} | Validation Cost: {cost_val:.4f}")
                
                if cost_val < best_val_cost:
                    best_val_cost, best_parameters, best_epoch, patience_counter = cost_val, parameters.copy(), i, 0
                    if print_cost: print(f"          -> (The best model has been saved.)")
                    FINAL_EPOCHS = best_epoch
                else:
                    patience_counter += 1
                    if print_cost: print(f"          -> (Patience: {patience_counter}/{patience})")
                
                if patience_counter >= patience:
                    print(f"\n--- Early Stopping: {best_epoch} (Validation Cost: {best_val_cost:.4f})")
                    break 
            else:
                validation_costs.append(np.nan) 
                if print_cost: print(f"Epoch {i} | Training Cost: {cost:.4f}")
            
    if do_validation and best_parameters:
        return best_parameters, costs, validation_costs
    
    return parameters, costs, validation_costs

In [ ]:
# ===============================================
# H) MAIN CONTROL AND TRAINING BLOCK (UPDATED FOR 2 LAYERS)
# ===============================================

print(f"\n--- Starting Model Training ---")
# UPDATED: N_H -> N_H1, N_H2
print(f"Input Size (N_X): {N_X} | Layer 1 (N_H1): {N_H1} | Layer 2 (N_H2): {N_H2}") 
print(f"Learning Rate: {LEARNING_RATE} | Patience: {PATIENCE} | L2 Penalty (Lambda): {LAMBDA}")
print("-" * 30)

# Ask the user what they want to do
print("Which mode would you like to run?")
print("  1: Phase 1 - Find the best epoch (Validation and Early Stopping Active)")
print("  2: Phase 2 - Train the final model (Validation Off, All Data Used)")
print("-" * 30)
user_choice = input("Please make your choice (1 or 2): ")

# Ask the user which method they want to use
print("Which model would you like to run?")
print(" 1 - Tanh activation function with Xavier initialization? (Slow)")
print(" 2 - Leaky ReLU activation function with He initialization? (Fast - Recommended)")
print("-" * 30)
model_choice = input("Please make your choice (1 or 2): ")

if (model_choice == '1'):
    MODEL_CHOICE = 'Tanh/Xavier'
elif (model_choice == '2'):
    MODEL_CHOICE = 'Leaky ReLU/He'
else:
    print("Invalid choice. Stopping the program.")
    exit()

# --- Define variables with default values ---
optimized_parameters = {}
costs = []
validation_costs = []
plot_title = ""

if user_choice == '1':
    # --- PHASE 1: TRAINING WITH VALIDATION (EARLY STOPPING) ---
    print("\n--- STARTING PHASE 1: Finding the Best Epoch ---")
    
    if model_choice == '1':
        print("\n--- Starting training with validation using Tanh with Xavier ---")
        optimized_parameters, costs, validation_costs = train_ann_tanh_xavier( 
            X=X,  # 1968-2021
            Y=Y,  # 1968-2021
            n_h1=N_H1, n_h2=N_H2, # UPDATED
            learning_rate=LEARNING_RATE, 
            num_epochs=EPOCHS, 
            X_val=X_val,  # 2022-2023 (Active)
            Y_val=Y_val,  # 2022-2023 (Active)
            print_cost=True,
            patience=PATIENCE,
            lambd=LAMBDA # UPDATED (For consistency)
        )
        
    elif model_choice == '2':
        print("\n--- Starting training with validation using Leaky ReLU with He ---")
        optimized_parameters, costs, validation_costs = train_ann_LeakyReLU_He( 
            X=X,  # 1968-2021
            Y=Y,  # 1968-2021
            n_h1=N_H1, n_h2=N_H2, # UPDATED
            learning_rate=LEARNING_RATE, 
            num_epochs=EPOCHS, 
            X_val=X_val,  # 2022-2023 (Active)
            Y_val=Y_val,  # 2022-2023 (Active)
            print_cost=True,
            patience=PATIENCE,
            lambd=LAMBDA
        )

    plot_title = f"Phase 1: Cost (LR: {LEARNING_RATE}, H1: {N_H1}, H2: {N_H2})" # UPDATED
    print(f"\n--- PHASE 1 COMPLETED. Please note the graph and the 'best epoch' number. ---")

elif user_choice == '2':
    # --- PHASE 2: FINAL MODEL TRAINING ---
    print("\n--- STARTING PHASE 2: Training the Final Model ---")
    
    # Ask the user for the 'best epoch' number they found in Phase 1
    try:
        best_epoch_str = input(f"Enter the 'best' epoch number you found in Phase 1 (e.g., 1350): ")
        BEST_EPOCH = int(best_epoch_str)
        FINAL_EPOCHS = BEST_EPOCH
        if BEST_EPOCH <= 0: raise ValueError
    except ValueError:
        print(f"Invalid number. Will use default {EPOCHS} epochs.")
        BEST_EPOCH = EPOCHS
        FINAL_EPOCHS = EPOCHS

    if model_choice == '1':
        print("\n--- Starting training without validation using Tanh with Xavier ---")
        optimized_parameters, costs, validation_costs = train_ann_tanh_xavier( 
            X=X_final_train,  # 1968-2023 (Combined)
            Y=Y_final_train,  # 1968-2023 (Combined)
            n_h1=N_H1, n_h2=N_H2, # UPDATED
            learning_rate=LEARNING_RATE, 
            num_epochs=BEST_EPOCH, # Only the 'best' found epoch count
            X_val=None,  # <-- DISABLED
            Y_val=None,  # <-- DISABLED
            print_cost=True,
            patience=9999, # (Ineffective)
            lambd=LAMBDA # UPDATED (For consistency)
        )
        
    elif model_choice == '2':
        print("\n--- Starting training without validation using Leaky ReLU with He ---")
        optimized_parameters, costs, validation_costs = train_ann_LeakyReLU_He( 
            X=X_final_train,  # 1968-2023 (Combined)
            Y=Y_final_train,  # 1968-2023 (Combined)
            n_h1=N_H1, n_h2=N_H2, # UPDATED
            learning_rate=LEARNING_RATE, 
            num_epochs=BEST_EPOCH, 
            X_val=None,  # <-- DISABLED
            Y_val=None,  # <-- DISABLED
            print_cost=True,
            patience=9999,
            lambd=LAMBDA
        )
    
    plot_title = f"Phase 2: Cost (LR: {LEARNING_RATE}, H1: {N_H1}, H2: {N_H2})" # UPDATED
    print(f"\n--- PHASE 2 (FINAL TRAINING) COMPLETED. ---")
    
else:
    print("Invalid choice. Stopping the program.")
    # (Program stops, 'optimized_parameters' remains empty)

In [ ]:
# ===============================================
# I) VISUALIZATION AND FINAL PERFORMANCE
# ===============================================

# 1. VISUALIZATION
# (If training was done, draw the graph)
if optimized_parameters:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 6))

    # Dynamically create the X-axis based on the actual length of the 'costs' list
    print_interval = 10
    # Get the actual length of the 'costs' list (may have stopped early)
    num_points_in_plot = len(costs) 
    epochs_list_actual = np.arange(0, num_points_in_plot) * print_interval

    # Training Cost
    plt.plot(epochs_list_actual, costs, label="Training Cost")
    
    # Validation Cost (Only drawn in Phase 1)
    # (In Phase 2, the 'validation_costs' list will be full of NaN)
    plt.plot(epochs_list_actual, validation_costs, label="Validation Cost", linestyle='--')

    plt.title(plot_title)
    plt.xlabel("Epoch")
    plt.ylabel("Cost (J)")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# J) ACCURACY CALCULATION FUNCTION
def evaluate_accuracy(Y_prediction, Y_true):
    """
    Calculates the model's accuracy score by comparing predictions (Y_prediction) 
    with true labels (Y_true).
    """
    # In NumPy, == is used to compare two matrices. 
    # The result is a matrix of (True=1, False=0).
    correct_predictions = (Y_prediction == Y_true)
    
    # Taking the mean directly gives us the ratio of correct predictions (Accuracy).
    accuracy = np.mean(correct_predictions) 
    
    return accuracy


# I) PREDICTION FUNCTION
def predict(parameters, X, model_choice):
    """
    Makes predictions for input X using trained parameters.
    
    Arguments:
    parameters -- Dictionary of trained W and b parameters
    X -- Input matrix to predict (N, m)
    
    Returns:
    Y_prediction -- NumPy vector of 0 or 1 labels for input X
    """
    
    if model_choice == '1':
        A2, cache = forward_propagation_tanh(X, parameters)
    elif model_choice == '2':
        A2, cache = forward_propagation_LeakyReLU(X, parameters)    
    
    # Decision Threshold: If probability is greater than 0.5, it's 1 (won), otherwise 0 (lost).
    Y_prediction = (A2 > 0.5).astype(int)
    
    return Y_prediction

# ===============================================
# I.5) PROBABILISTIC PREDICTION FUNCTION (NEW)
# ===============================================

def predict_with_proba(parameters, X, model_choice):
    """
    Returns both the final decision (0/1) and the raw probability (A_out)
    using trained parameters.
    """
    A_out, cache = None, None
    
    # Call the correct forward_propagation function (1 or 2 layer)
    # based on model choice
    if model_choice == '1':
        # (If 1-layer model)
        # A_out, cache = forward_propagation_tanh_1_layer(X, parameters) 
        # (If 2-layer model)
        A_out, cache = forward_propagation_tanh(X, parameters) 
    elif model_choice == '2':
        # (If 1-layer model)
        # A_out, cache = forward_propagation_LeakyReLU_1_layer(X, parameters)
        # (If 2-layer model)
        A_out, cache = forward_propagation_LeakyReLU(X, parameters)
    
    # Decision Threshold
    Y_prediction = (A_out > 0.5).astype(int)
    
    # RETURN BOTH VALUES
    return Y_prediction, A_out

In [ ]:
# ===============================================
# K) FINAL PERFORMANCE AND DETAILED MATCH ANALYSIS
# ===============================================

# 1. Test Set Prediction (NEW: With Probabilities)
Y_pred_test, A_test_proba = predict_with_proba(optimized_parameters, X_test, model_choice)

# 2. Overall Accuracy Calculation
test_accuracy = evaluate_accuracy(Y_pred_test, Y_test)

print(f"\n{'='*60}")
print(f"MODEL PERFORMANCE REPORT (Feature-Engineered Dataset)")
print(f"{'='*60}")
print(f"Test Set (2024) Sample Count: {X_test.shape[1]}")
print(f"Number of Features Used: {N_X} (51 engineered features)")
print(f"Model Architecture: {N_X} → {N_H1} → {N_H2} → 1")
print(f"Activation: {'Leaky ReLU' if model_choice == '2' else 'Tanh'}")
print(f"Normalization: StandardScaler (mean=0, std=1)")
print("-" * 60)
print(f"🎯 FINAL TEST ACCURACY (2024): {test_accuracy*100:.2f}%")
print("-" * 60)

# Comparison with baseline (if you have previous results)
print(f"\n📊 Expected Improvement:")
print(f"   • Basic features (9 features): ~60-65% accuracy")
print(f"   • Feature-engineered (51 features): ~70-75% accuracy")
print(f"   • Actual result: {test_accuracy*100:.2f}%")

# 3. Detailed Match Analysis
print(f"\n{'='*60}")
print(f"DETAILED MATCH ANALYSIS (Test Set)")
print(f"{'='*60}")

# Flatten data for easier analysis (make it 1D)
true_labels = Y_test.flatten()
pred_labels = Y_pred_test.flatten()
p1_win_probabilities = A_test_proba.flatten()

# Create a DataFrame for analysis
df_results = pd.DataFrame({
    'Match Index': np.arange(len(true_labels)),
    'Actual Winner': np.where(true_labels == 1, 'P1', 'P2'),
    'Predicted': np.where(pred_labels == 1, 'P1', 'P2'),
    'P1 Winning Probability (%)': p1_win_probabilities * 100,
    'Result': np.where(true_labels == pred_labels, 'CORRECT', 'WRONG')
})

# Calculate the "Confidence Percentage" in the model's prediction
df_results['Prediction Confidence (%)'] = np.where(
    df_results['Predicted'] == 'P1', 
    df_results['P1 Winning Probability (%)'], 
    100.0 - df_results['P1 Winning Probability (%)']
)

# Sort results by 'Prediction Confidence'
df_results_sorted = df_results.sort_values(by='Prediction Confidence (%)', ascending=False)

# --- Analysis Outputs ---

print("\n🎯 Top 10 CORRECT Predictions with Highest Confidence:")
print("-" * 60)
top_10_correct = df_results_sorted[df_results_sorted['Result'] == 'CORRECT'].head(10)
print(top_10_correct[['Match Index', 'Predicted', 'Prediction Confidence (%)', 'Actual Winner']].to_string(index=False))

print(f"\n❌ Top 10 WRONG Predictions with Highest Confidence (Worst Errors):")
print("-" * 60)
top_10_incorrect = df_results_sorted[df_results_sorted['Result'] == 'WRONG'].head(10)
print(top_10_incorrect[['Match Index', 'Predicted', 'Prediction Confidence (%)', 'Actual Winner']].to_string(index=False))

print(f"\n🤔 Top 10 Predictions with Lowest Confidence (Coin-Flip Matches):")
print("-" * 60)
top_10_unsure = df_results.iloc[
    (df_results['Prediction Confidence (%)'] - 50).abs().argsort()
].head(10)
print(top_10_unsure[['Match Index', 'Predicted', 'Prediction Confidence (%)', 'Actual Winner', 'Result']].to_string(index=False))

# Accuracy analysis by confidence intervals
print(f"\n📈 Accuracy by Confidence Intervals:")
print("-" * 60)
confidence_bins = [50, 60, 70, 80, 90, 100]
for i in range(len(confidence_bins)-1):
    low, high = confidence_bins[i], confidence_bins[i+1]
    mask = (df_results['Prediction Confidence (%)'] >= low) & (df_results['Prediction Confidence (%)'] < high)
    subset = df_results[mask]
    if len(subset) > 0:
        acc = (subset['Result'] == 'CORRECT').mean() * 100
        print(f"   {low}%-{high}% confidence: {len(subset):4d} matches, {acc:.1f}% accuracy")

print(f"\n{'='*60}")
print(f"✅ ANALYSIS COMPLETED!")
print(f"{'='*60}")

In [ ]:
# ===============================================
# K) FINAL PERFORMANCE AND SELECTIVE ACCURACY ANALYSIS
# ===============================================

# Import analysis and visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Test Set Prediction (with Probabilities)
# (Assumes 'optimized_parameters' was trained in block H)
if 'optimized_parameters' in locals() and optimized_parameters:
    
    # Get Y_pred_test (0/1) AND A_test_proba (0.0-1.0) values
    Y_pred_test, A_test_proba = predict_with_proba(optimized_parameters, X_test, model_choice)

    # 2. Overall Accuracy Calculation (ALL MATCHES)
    test_accuracy = evaluate_accuracy(Y_pred_test, Y_test)

    print(f"\n--- Model Evaluation (Overall Result) ---")
    print(f"Test Set (2024) Sample Count: {X_test.shape[1]}")
    print("-" * 35)
    print(f"Final TEST Accuracy (All Matches): {test_accuracy*100:.2f}%")
    print("-" * 35)

    # 3. Create DataFrame for Detailed Match Analysis
    true_labels = Y_test.flatten()
    pred_labels = Y_pred_test.flatten()
    p1_win_probabilities = A_test_proba.flatten() 

    df_results = pd.DataFrame({
        'Actual Winner': np.where(true_labels == 1, 'P1', 'P2'),
        'Predicted': np.where(pred_labels == 1, 'P1', 'P2'),
        'P1 Winning Probability (%)': p1_win_probabilities * 100,
        'Result': np.where(true_labels == pred_labels, 'CORRECT', 'WRONG')
    })
    df_results['Prediction Confidence (%)'] = np.where(
        df_results['Predicted'] == 'P1', 
        df_results['P1 Winning Probability (%)'], 
        100.0 - df_results['P1 Winning Probability (%)']
    )
    
    # --- 4. NEW: SELECTIVE ACCURACY (PER YOUR REQUEST) ---
    print("\n--- Selective Accuracy (Confidence Thresholding) ---")
    
    # Define threshold (e.g., 75% confidence, i.e., ±25 points from 50)
    CONFIDENCE_MARGIN = 5  # (Setting 75 - 50 = 25)
    
    CONFIDENCE_THRESHOLD_UPPER = 50 + CONFIDENCE_MARGIN  # 75.0
    CONFIDENCE_THRESHOLD_LOWER = 50 - CONFIDENCE_MARGIN  # 25.0

    print(f"Only matches where model predicts P1 > %{CONFIDENCE_THRESHOLD_UPPER} or P1 < %{CONFIDENCE_THRESHOLD_LOWER} (P2 > %{CONFIDENCE_THRESHOLD_UPPER}) are considered...")

    # Filter only "confident" predictions
    confident_predictions_df = df_results[
        (df_results['P1 Winning Probability (%)'] > CONFIDENCE_THRESHOLD_UPPER) |
        (df_results['P1 Winning Probability (%)'] < CONFIDENCE_THRESHOLD_LOWER)
    ]

    total_matches = len(df_results)
    confident_matches = len(confident_predictions_df)
    
    if confident_matches > 0:
        # Compute accuracy only for these 'confident' matches
        selective_accuracy = (confident_predictions_df['Result'] == 'CORRECT').mean()
        
        print(f"\n   -> Model was confident in {confident_matches} out of {total_matches} matches (%{100*confident_matches/total_matches:.1f}).")
        print(f"   -> NEW accuracy for these 'confident' matches: {selective_accuracy*100:.2f}%")
        print("-" * 35)
    else:
        print("   -> Model was not confident in any match at this threshold.")
        print("-" * 35)
        
    
    from pathlib import Path
    import re

    # 1) Prepare folder
    out_dir = Path('./confidence_imgs')
    out_dir.mkdir(parents=True, exist_ok=True)

    # 2) Base name and pattern
    base = 'confidence_correct_vs_incorrect_dist'
    numbered_regex = re.compile(rf'^{re.escape(base)}(\d+)\.png$')

    # 3) Scan existing files
    max_idx = 0
    bare_exists = (out_dir / f'{base}.png').exists()  # Is there an unnumbered file?

    if bare_exists:
        max_idx = 1  # Treat unnumbered file as "1"

    for p in out_dir.glob(f'{base}[0-9]*.png'):
        m = numbered_regex.match(p.name)
        if m:
            idx = int(m.group(1))
            if idx > max_idx:
                max_idx = idx

    # 4) New file name
    next_idx = max_idx + 1 if (bare_exists or max_idx > 0) else 1
    out_path = out_dir / f'{base}{next_idx}.png'


    # --- 5. CONFIDENCE DISTRIBUTION GRAPHS ---
    # (This code can remain unchanged from the previous version)
    print("\nCreating Graph 1 (Correct vs Wrong)...")
    plt.figure(figsize=(12, 7))
    bins_range = np.arange(50, 101, 5) # 50, 55, 60... 100
    sns.histplot(df_results[df_results['Result'] == 'CORRECT']['Prediction Confidence (%)'], bins=bins_range, kde=False, color='green', alpha=0.6, label='CORRECT Predictions')
    sns.histplot(df_results[df_results['Result'] == 'WRONG']['Prediction Confidence (%)'], bins=bins_range, kde=False, color='red', alpha=0.6, label='WRONG Predictions')
    plt.title('Prediction Confidence Distribution for Correct/Wrong (Test Set)')
    plt.xlabel('Model Confidence in Prediction (%)')
    plt.ylabel('Number of Matches')
    plt.legend()
    epochs_disp = FINAL_EPOCHS if FINAL_EPOCHS else EPOCHS
    print(FINAL_EPOCHS)
    ax = plt.gca()
    DATA_PATH = DATA_PATH
    ax.text(
        0.99, 0.99,
        f"Data: {DATA_PATH}\n"
        f"Model: {MODEL_CHOICE}\n"
        f"H1={N_H1}, H2={N_H2}\n"
        f"LR={LEARNING_RATE}, L2={LAMBDA}\n"
        f"Epochs={epochs_disp}, Patience={PATIENCE}",
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='white', ec='#dddddd', alpha=0.85)
    )
    print(f'File to be saved: {out_path}')
    plt.grid(True)
    plt.savefig(out_path)
    plt.show()

else:
    print("\nPLEASE FIRST RUN BLOCK H TO TRAIN A MODEL.")
    print("Trained 'optimized_parameters' are required for analysis.")


In [ ]:
# ===============================================
# L) WIMBLEDON 2025 PREDICTIONS (WITH IBM COMPARISON)
# ===============================================

from pathlib import Path

print("="*80)
print("🎾 WIMBLEDON 2025 - MODEL PREDICTIONS vs IBM")
print("="*80)

# Load IBM predictions
ibm_df = pd.read_csv("IBM's_predictions.csv")
print(f"✓ Loaded {len(ibm_df)} IBM predictions")

# MODEL FEATURE ORDER (from training)
MODEL_FEATURES = [
    'both_left_handed', 'both_right_handed', 'h2h_matches', 'h2h_win_rate',
    'mixed_handed', 'month', 'month_cos', 'month_sin', 'p1_age',
    'p1_avg_1st_serve_pct', 'p1_avg_1st_serve_win_pct', 'p1_avg_ace_rate',
    'p1_avg_df_rate', 'p1_career_win_rate', 'p1_experience', 'p1_ht',
    'p1_is_lefty', 'p1_rank_points', 'p1_recent_form', 'p1_surface_win_rate',
    'p2_age', 'p2_avg_1st_serve_pct', 'p2_avg_1st_serve_win_pct',
    'p2_avg_ace_rate', 'p2_avg_df_rate', 'p2_career_win_rate', 'p2_experience',
    'p2_ht', 'p2_is_lefty', 'p2_rank_points', 'p2_recent_form',
    'p2_surface_win_rate', 'quarter', 'surface_clay', 'surface_grass',
    'surface_hard', 'tourney_importance'
]

print(f"✓ Model expects {len(MODEL_FEATURES)} features in this exact order")

# Rounds folder path
rounds_folder = Path('2025Wimbledon/rounds')

# Round files
round_files = {
    'Round 1': 'Round1_FirstRound.csv',
    'Round 2': 'Round2_SecondRound.csv',
    'Round 3': 'Round3_ThirdRound.csv',
    'Round 4': 'Round4_FourthRound.csv',
    'Quarter Finals': 'QuarterFinals.csv',
    'Semi Finals': 'SemiFinals.csv',
    'Final': 'Final.csv'
}

# Function to predict matches for a specific round
def predict_wimbledon_round(round_file, round_name, ibm_predictions, match_offset):
    """Predict matches for a specific round and compare with IBM"""
    
    # Load CSV
    df = pd.read_csv(rounds_folder / round_file)
    
    # CRITICAL: Extract features in MODEL ORDER!
    X = df[MODEL_FEATURES].values  # (n_matches, 37) in CORRECT order
    
    # Extract metadata (for comparison) - NOT given to model
    metadata = df[['player1_name', 'player2_name', 'winner']].copy()
    
    # Transpose for model format (features × samples)
    X_T = X.T  # (37, n_matches)
    
    # PREDICT using your trained model
    Y_pred, A_proba = predict_with_proba(optimized_parameters, X_T, model_choice)
    
    # Flatten
    probabilities = A_proba.flatten()  # P(P1 wins)
    
    # Calculate MY MODEL probabilities
    my_p1_probs = probabilities * 100      # P(P1 wins)
    my_p2_probs = (1 - probabilities) * 100  # P(P2 wins)
    
    # MY MODEL predicted winner: whoever has higher probability
    my_predicted_winners = np.where(my_p1_probs > my_p2_probs, 1, 2)
    
    # MY MODEL confidence = probability of the predicted winner
    my_confidences = np.where(my_predicted_winners == 1, my_p1_probs, my_p2_probs)
    
    # Create results dataframe
    results = metadata.copy()
    results['my_predicted_winner'] = my_predicted_winners
    results['my_p1_probability'] = my_p1_probs
    results['my_p2_probability'] = my_p2_probs
    results['my_confidence'] = my_confidences
    results['my_correct'] = (results['my_predicted_winner'] == results['winner'])
    
    # Add IBM predictions
    num_matches = len(results)
    ibm_slice = ibm_predictions.iloc[match_offset:match_offset + num_matches]
    
    results['ibm_p1_probability'] = ibm_slice['P1%'].values
    results['ibm_p2_probability'] = ibm_slice['P2%'].values
    
    # IBM predicted winner
    results['ibm_predicted_winner'] = np.where(
        results['ibm_p1_probability'] > results['ibm_p2_probability'], 1, 2
    )
    
    # IBM confidence
    results['ibm_confidence'] = np.where(
        results['ibm_predicted_winner'] == 1,
        results['ibm_p1_probability'],
        results['ibm_p2_probability']
    )
    
    # IBM correct
    results['ibm_correct'] = (results['ibm_predicted_winner'] == results['winner'])
    
    # Calculate accuracies
    my_accuracy = results['my_correct'].mean()
    ibm_accuracy = results['ibm_correct'].mean()
    my_correct_count = results['my_correct'].sum()
    ibm_correct_count = results['ibm_correct'].sum()
    total_count = len(results)
    
    # Display results
    print(f"\n{'='*130}")
    print(f"📊 {round_name}")
    print(f"{'='*130}")
    print(f"MY MODEL Accuracy: {my_accuracy:.1%} ({my_correct_count}/{total_count}) | IBM Accuracy: {ibm_accuracy:.1%} ({ibm_correct_count}/{total_count})")
    print(f"\n{'Player 1':<20s} {'Player 2':<20s} {'Act':<4s} {'MyP':<4s} {'MyP1%':<6s} {'MyP2%':<6s} {'MyCf':<6s} {'✓':<2s} {'IBM':<4s} {'IP1%':<6s} {'IP2%':<6s} {'ICf':<6s} {'✓':<2s}")
    print("-"*130)
    
    for idx, row in results.iterrows():
        my_check = "✓" if row['my_correct'] else "✗"
        ibm_check = "✓" if row['ibm_correct'] else "✗"
        
        my_p1 = f"{row['my_p1_probability']:.1f}%"
        my_p2 = f"{row['my_p2_probability']:.1f}%"
        my_cf = f"{row['my_confidence']:.1f}%"
        
        ibm_p1 = f"{row['ibm_p1_probability']:.0f}%"
        ibm_p2 = f"{row['ibm_p2_probability']:.0f}%"
        ibm_cf = f"{row['ibm_confidence']:.0f}%"
        
        print(f"{row['player1_name']:<20s} {row['player2_name']:<20s} "
              f"P{int(row['winner']):<3d} "
              f"P{int(row['my_predicted_winner']):<3d} {my_p1:<6s} {my_p2:<6s} {my_cf:<6s} {my_check:<2s} "
              f"P{int(row['ibm_predicted_winner']):<3d} {ibm_p1:<6s} {ibm_p2:<6s} {ibm_cf:<6s} {ibm_check:<2s}")
    
    return results, my_accuracy, ibm_accuracy, num_matches

# Predict all rounds
all_wimbledon_results = {}
my_total_correct = 0
ibm_total_correct = 0
total_matches = 0
match_offset = 0

for round_name, round_file in round_files.items():
    file_path = rounds_folder / round_file
    
    if file_path.exists():
        results, my_acc, ibm_acc, num_matches = predict_wimbledon_round(
            round_file, round_name, ibm_df, match_offset
        )
        all_wimbledon_results[round_name] = results
        
        my_total_correct += results['my_correct'].sum()
        ibm_total_correct += results['ibm_correct'].sum()
        total_matches += num_matches
        match_offset += num_matches
    else:
        print(f"⚠️  {round_file} not found, skipping...")

# Overall statistics
print(f"\n{'='*130}")
print(f"📈 OVERALL WIMBLEDON 2025 RESULTS - MY MODEL vs IBM")
print(f"{'='*130}")
print(f"Total matches: {total_matches}")
print(f"\nMY MODEL:")
print(f"  Correct predictions: {my_total_correct}")
print(f"  Overall accuracy: {my_total_correct/total_matches:.1%}")
print(f"\nIBM:")
print(f"  Correct predictions: {ibm_total_correct}")
print(f"  Overall accuracy: {ibm_total_correct/total_matches:.1%}")
print(f"\nDIFFERENCE:")
diff = my_total_correct - ibm_total_correct
diff_pct = (my_total_correct/total_matches - ibm_total_correct/total_matches) * 100
if diff > 0:
    print(f"  MY MODEL is BETTER by {diff} matches ({diff_pct:+.1f}%)")
elif diff < 0:
    print(f"  IBM is BETTER by {-diff} matches ({diff_pct:+.1f}%)")
else:
    print(f"  TIED!")

# Accuracy by round comparison
print(f"\n📊 Accuracy by Round - MY MODEL vs IBM:")
print("-"*80)
print(f"{'Round':<20s} {'My Model':<20s} {'IBM':<20s} {'Difference':<20s}")
print("-"*80)
for round_name, results in all_wimbledon_results.items():
    my_acc = results['my_correct'].mean()
    ibm_acc = results['ibm_correct'].mean()
    count = len(results)
    my_correct = results['my_correct'].sum()
    ibm_correct = results['ibm_correct'].sum()
    diff = my_correct - ibm_correct
    diff_pct = (my_acc - ibm_acc) * 100
    
    my_str = f"{my_acc:.1%} ({my_correct}/{count})"
    ibm_str = f"{ibm_acc:.1%} ({ibm_correct}/{count})"
    diff_str = f"{diff_pct:+.1f}% ({diff:+d})"
    
    print(f"{round_name:<20s} {my_str:<20s} {ibm_str:<20s} {diff_str:<20s}")

print(f"\n{'='*130}")
print(f"✅ WIMBLEDON 2025 PREDICTION COMPLETE!")
print(f"{'='*130}")